In [1]:
# ============================================================
# COST-SENSITIVE RL FOR EARLY SAE WARNING
# ============================================================

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Project root
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if PROJECT_ROOT.name == "test":
    PROJECT_ROOT = PROJECT_ROOT.parent

print("Project root:", PROJECT_ROOT)

# Add project root to Python path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "mimic3"
    / "clinical_hourly_v1.csv"
)

print("Dataset path:", DATA_PATH)
print("Exists:", DATA_PATH.exists())

Project root: /Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning
Dataset path: /Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning/data/processed/mimic3/clinical_hourly_v1.csv
Exists: True


In [2]:
# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nSAE distribution:")
print(df["sae"].value_counts())

print("\nICU stays:", df["icustay_id"].nunique())
print("Patients:", df["subject_id"].nunique())

Dataset shape: (4556, 17)

Columns:
['subject_id', 'hadm_id', 'icustay_id', 'hour', 'gcs_eye', 'gcs_motor', 'gcs_verbal', 'heart_rate', 'map', 'resp_rate', 'spo2', 'gcs_total', 'previous_gcs', 'gcs_change', 'gcs_last_observed', 'previous_observed_gcs', 'sae']

SAE distribution:
sae
0    4521
1      35
Name: count, dtype: int64

ICU stays: 38
Patients: 25


In [3]:
# ============================================================
# FUTURE 1-HOUR SAE TARGET
# ============================================================

df = df.sort_values(
    ["icustay_id", "hour"]
).reset_index(drop=True)

df["future_sae_1h"] = (
    df.groupby("icustay_id")["sae"]
      .shift(-1)
      .fillna(0)
      .astype(int)
)

print("Current SAE:")
print(df["sae"].value_counts())

print("\nFuture 1-hour SAE:")
print(df["future_sae_1h"].value_counts())

print(
    "\nFuture SAE events:",
    df["future_sae_1h"].sum()
)

Current SAE:
sae
0    4521
1      35
Name: count, dtype: int64

Future 1-hour SAE:
future_sae_1h
0    4521
1      35
Name: count, dtype: int64

Future SAE events: 35


In [4]:
# ============================================================
# PATIENT-LEVEL TRAIN / TEST SPLIT
# ============================================================

patients = df["subject_id"].unique()

rng = np.random.default_rng(42)
rng.shuffle(patients)

n_train = int(len(patients) * 0.80)

train_patients = patients[:n_train]
test_patients = patients[n_train:]

train_df = df[df["subject_id"].isin(train_patients)].copy()
test_df = df[df["subject_id"].isin(test_patients)].copy()

print("TRAIN")
print("Patients:", train_df["subject_id"].nunique())
print("ICU stays:", train_df["icustay_id"].nunique())
print("Rows:", len(train_df))
print("Future SAE:", train_df["future_sae_1h"].sum())

print("\nTEST")
print("Patients:", test_df["subject_id"].nunique())
print("ICU stays:", test_df["icustay_id"].nunique())
print("Rows:", len(test_df))
print("Future SAE:", test_df["future_sae_1h"].sum())

print(
    "\nPatient overlap:",
    len(
        set(train_df["subject_id"])
        &
        set(test_df["subject_id"])
    )
)

TRAIN
Patients: 20
ICU stays: 33
Rows: 3302
Future SAE: 22

TEST
Patients: 5
ICU stays: 5
Rows: 1254
Future SAE: 13

Patient overlap: 0


In [5]:
# ============================================================
# PREPROCESSING
# ============================================================

FEATURES = [
    "gcs_last_observed",
    "previous_observed_gcs",
    "heart_rate",
    "map",
    "resp_rate",
    "spo2",
    "hour"
]

TARGET = "future_sae_1h"

# Median values learned ONLY from training data
train_medians = train_df[FEATURES].median()

train_df[FEATURES] = train_df[FEATURES].fillna(train_medians)
test_df[FEATURES] = test_df[FEATURES].fillna(train_medians)

print("Remaining train missing values:")
print(train_df[FEATURES].isna().sum())

print("\nRemaining test missing values:")
print(test_df[FEATURES].isna().sum())

Remaining train missing values:
gcs_last_observed        0
previous_observed_gcs    0
heart_rate               0
map                      0
resp_rate                0
spo2                     0
hour                     0
dtype: int64

Remaining test missing values:
gcs_last_observed        0
previous_observed_gcs    0
heart_rate               0
map                      0
resp_rate                0
spo2                     0
hour                     0
dtype: int64


In [6]:
# ============================================================
# COST-SENSITIVE SAE RL ENVIRONMENT
# ============================================================

import gymnasium as gym
from gymnasium import spaces


class CostSensitiveSAEEnv(gym.Env):

    def __init__(
        self,
        dataframe,
        miss_penalty=-10.0,
        hit_reward=10.0,
        false_alarm_penalty=-1.0,
        correct_negative_reward=0.1
    ):

        super().__init__()

        self.df = dataframe.reset_index(drop=True)

        self.features = FEATURES
        self.target = TARGET

        self.miss_penalty = miss_penalty
        self.hit_reward = hit_reward
        self.false_alarm_penalty = false_alarm_penalty
        self.correct_negative_reward = correct_negative_reward

        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(len(self.features),),
            dtype=np.float32
        )

        # 0 = no warning
        # 1 = moderate warning
        # 2 = high-risk warning
        self.action_space = spaces.Discrete(3)

        self.current_index = 0

    def _get_state(self):

        state = self.df.loc[
            self.current_index,
            self.features
        ].to_numpy(dtype=np.float32)

        return state

    def reset(self, seed=None, options=None):

        super().reset(seed=seed)

        self.current_index = 0

        state = self._get_state()

        info = {
            "icustay_id": int(
                self.df.loc[self.current_index, "icustay_id"]
            ),
            "hour": int(
                self.df.loc[self.current_index, "hour"]
            )
        }

        return state, info

    def step(self, action):

        action = int(action)

        true_event = int(
            self.df.loc[self.current_index, self.target]
        )

        # Any warning action counts as predicting SAE.
        predicted_event = 1 if action > 0 else 0

        # Cost-sensitive reward
        if true_event == 1 and predicted_event == 1:
            reward = self.hit_reward

        elif true_event == 1 and predicted_event == 0:
            reward = self.miss_penalty

        elif true_event == 0 and predicted_event == 1:
            reward = self.false_alarm_penalty

        else:
            reward = self.correct_negative_reward

        current_info = {
            "icustay_id": int(
                self.df.loc[self.current_index, "icustay_id"]
            ),
            "hour": int(
                self.df.loc[self.current_index, "hour"]
            ),
            "sae": true_event,
            "action": action
        }

        self.current_index += 1

        terminated = self.current_index >= len(self.df)

        truncated = False

        if terminated:
            next_state = np.zeros(
                len(self.features),
                dtype=np.float32
            )
        else:
            next_state = self._get_state()

        return (
            next_state,
            float(reward),
            terminated,
            truncated,
            current_info
        )


print("Environment class created successfully.")

Environment class created successfully.


In [7]:
# ============================================================
# TEST ENVIRONMENT
# ============================================================

env = CostSensitiveSAEEnv(train_df)

state, info = env.reset(seed=42)

print("Observation space:", env.observation_space)
print("Action space:", env.action_space)

print("\nInitial state:")
print(state)

print("\nInitial info:")
print(info)

next_state, reward, terminated, truncated, info = env.step(0)

print("\nAfter action 0:")
print("Next state:", next_state)
print("Reward:", reward)
print("Terminated:", terminated)
print("Info:", info)

Observation space: Box(-inf, inf, (7,), float32)
Action space: Discrete(3)

Initial state:
[ 11.  11. 100.  71.  32.  93.   0.]

Initial info:
{'icustay_id': 201006, 'hour': 0}

After action 0:
Next state: [ 11.   11.  104.5  71.   33.5  94.    1. ]
Reward: 0.1
Terminated: False
Info: {'icustay_id': 201006, 'hour': 0, 'sae': 0, 'action': 0}


In [8]:
# ============================================================
# TRAIN COST-SENSITIVE DQN
# ============================================================

from stable_baselines3 import DQN

train_env = CostSensitiveSAEEnv(
    train_df,
    miss_penalty=-10.0,
    hit_reward=10.0,
    false_alarm_penalty=-1.0,
    correct_negative_reward=0.1
)

model = DQN(
    "MlpPolicy",
    train_env,
    learning_rate=5e-4,
    buffer_size=10000,
    learning_starts=500,
    batch_size=64,
    gamma=0.95,
    train_freq=4,
    target_update_interval=500,
    exploration_fraction=0.30,
    exploration_final_eps=0.05,
    verbose=1,
    seed=42
)

model.learn(total_timesteps=30000)

print("\nTraining completed.")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 3.3e+03  |
|    ep_rew_mean      | -701     |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 4        |
|    fps              | 931      |
|    time_elapsed     | 14       |
|    total_timesteps  | 13208    |
| train/              |          |
|    learning_rate    | 0.0005   |
|    loss             | 0.156    |
|    n_updates        | 3176     |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 3.3e+03  |
|    ep_rew_mean      | -360     |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 8        |
|    fps              | 885      |
|    time_elapsed     | 29       |
|    total_timesteps  | 26416    |
| train/              |        

In [9]:
# ============================================================
# SAVE MODEL
# ============================================================

MODEL_DIR = PROJECT_ROOT / "models"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_PATH = MODEL_DIR / "dqn_cost_sensitive_sae"

model.save(MODEL_PATH)

print("Model saved to:")
print(MODEL_PATH)

Model saved to:
/Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning/models/dqn_cost_sensitive_sae


In [10]:
# ============================================================
# EVALUATION
# ============================================================

test_env = CostSensitiveSAEEnv(test_df)

state, info = test_env.reset(seed=123)

y_true = []
y_pred = []

total_reward = 0.0

while True:

    action, _ = model.predict(
        np.asarray(state, dtype=np.float32),
        deterministic=True
    )

    action = int(np.asarray(action).item())

    true_event = int(
        test_env.df.loc[
            test_env.current_index,
            TARGET
        ]
    )

    predicted_event = 1 if action > 0 else 0

    y_true.append(true_event)
    y_pred.append(predicted_event)

    (
        next_state,
        reward,
        terminated,
        truncated,
        info
    ) = test_env.step(action)

    total_reward += reward

    state = next_state

    if terminated or truncated:
        break

precision = precision_score(
    y_true,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0
)

cm = confusion_matrix(
    y_true,
    y_pred
)

print("COST-SENSITIVE DQN RESULTS")
print("=" * 40)

print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

print("\nTotal reward:", total_reward)

print("\nConfusion matrix:")
print(cm)

print("\nClassification report:")
print(
    classification_report(
        y_true,
        y_pred,
        zero_division=0
    )
)

COST-SENSITIVE DQN RESULTS
Precision: 0.0
Recall: 0.0
F1: 0.0

Total reward: -32.29999999999854

Confusion matrix:
[[1217   24]
 [  13    0]]

Classification report:
              precision    recall  f1-score   support

           0       0.99      0.98      0.99      1241
           1       0.00      0.00      0.00        13

    accuracy                           0.97      1254
   macro avg       0.49      0.49      0.49      1254
weighted avg       0.98      0.97      0.97      1254



In [11]:
# ============================================================
# ACTION DISTRIBUTION
# ============================================================

actions = []

test_env = CostSensitiveSAEEnv(test_df)

state, info = test_env.reset(seed=123)

while True:

    action, _ = model.predict(
        np.asarray(state, dtype=np.float32),
        deterministic=True
    )

    action = int(np.asarray(action).item())

    actions.append(action)

    (
        state,
        reward,
        terminated,
        truncated,
        info
    ) = test_env.step(action)

    if terminated or truncated:
        break

print("Action distribution:")
print(
    pd.Series(actions)
    .value_counts()
    .sort_index()
)

print("\nAction percentages:")
print(
    pd.Series(actions)
    .value_counts(normalize=True)
    .sort_index() * 100
)

Action distribution:
0    1230
2      24
Name: count, dtype: int64

Action percentages:
0    98.086124
2     1.913876
Name: proportion, dtype: float64


In [12]:
# ============================================================
# FINAL COMPARISON
# ============================================================

results = pd.DataFrame({
    "Model": [
        "Original DQN",
        "Reward-Shaped DQN",
        "Synthetic-Augmented DQN",
        "Cost-Sensitive DQN"
    ],
    "Precision": [
        0.058824,
        0.0,
        0.0,
        precision
    ],
    "Recall": [
        0.333333,
        0.0,
        0.0,
        recall
    ],
    "F1": [
        0.100000,
        0.0,
        0.0,
        f1
    ],
    "Total Reward": [
        82.0,
        1176.0,
        1174.5,
        total_reward
    ]
})

print(results)

                     Model  Precision    Recall   F1  Total Reward
0             Original DQN   0.058824  0.333333  0.1          82.0
1        Reward-Shaped DQN   0.000000  0.000000  0.0        1176.0
2  Synthetic-Augmented DQN   0.000000  0.000000  0.0        1174.5
3       Cost-Sensitive DQN   0.000000  0.000000  0.0         -32.3


In [13]:
# ============================================================
# SAVE FINAL RESULTS
# ============================================================

RESULTS_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "mimic3"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULT_PATH = (
    RESULTS_DIR
    / "sae_rl_cost_sensitive_results.csv"
)

results.to_csv(
    RESULT_PATH,
    index=False
)

print("Results saved:")
print(RESULT_PATH)

Results saved:
/Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning/data/processed/mimic3/sae_rl_cost_sensitive_results.csv
